# Confronto metriche `*_dist_nll_corr` e `*_sep_corr` su N history.json

Imposta una lista di percorsi a file `history.json`, poi esegui tutte le celle.
Ogni subplot mostra una metrica confrontando tutti i path forniti.

In [ ]:
import json
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import torch

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    base = (start or Path.cwd()).resolve()
    for candidate in [base, *base.parents]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Project root non trovato (attesi folder 'src' e 'data').")


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.dpo_config import Config
from src.dpo_data import pad_encode, read_fasta
from src.dpo_metrics import compute_sequence_nll
from src.dpo_plotting import (
    _build_binary_labels_and_scores,
    _compute_pr_curve,
    _compute_roc_curve,
)
from src.transformer import GPTTransformer

cfg = Config()
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

split_dir = project_root / "data/split_train_validation/split_fraction-0.15_add-vae-25-30-and-close-to-validation_nll-filtered_100"
dn_path = project_root / cfg.dn_path
pretrained_ckpt_path = project_root / cfg.pretrained_ckpt_path

if not dn_path.exists():
    raise FileNotFoundError(f"DN FASTA non trovato: {dn_path}")
if not pretrained_ckpt_path.exists():
    raise FileNotFoundError(f"Checkpoint pretrained non trovato: {pretrained_ckpt_path}")

train_good_path = split_dir / "DTp_train.fasta"
train_bad_path = split_dir / "DTm_train_after-nll-filter.fasta"
val_good_path = split_dir / "DTp_val_plus-vae25-30-and-close.fasta"
val_bad_path = split_dir / "DTm_val_plus-vae25-30-and-close.fasta"
val_1_good_path = split_dir / "DTp_vae25-30_plus-close-close.fasta"
val_1_bad_path = split_dir / "DTm_vae25-30_plus-close-close.fasta"

for p in [
    train_good_path,
    train_bad_path,
    val_good_path,
    val_bad_path,
    val_1_good_path,
    val_1_bad_path,
]:
    if not p.exists():
        raise FileNotFoundError(f"File split non trovato: {p}")

with dn_path.open("r", encoding="utf-8") as f:
    dn_lines = f.read().splitlines()[1::2]
chars = sorted(list(set("\n".join(dn_lines))))
pad_symbol = "?"
chars = chars + [pad_symbol]
stoi = {ch: i for i, ch in enumerate(chars)}

pad_token = stoi[pad_symbol]
vocab_size = len(stoi)


def encode(s: str) -> list[int]:
    return [stoi[c] for c in s]


train_good_sequences = read_fasta(train_good_path)
train_bad_sequences = read_fasta(train_bad_path)
val_good_sequences = read_fasta(val_good_path)
val_bad_sequences = read_fasta(val_bad_path)
val_1_good_sequences = read_fasta(val_1_good_path)
val_1_bad_sequences = read_fasta(val_1_bad_path)

train_good_data = [pad_encode(s, cfg.block_size, pad_token, encode) for s in train_good_sequences]
train_bad_data = [pad_encode(s, cfg.block_size, pad_token, encode) for s in train_bad_sequences]
val_good_data = [pad_encode(s, cfg.block_size, pad_token, encode) for s in val_good_sequences]
val_bad_data = [pad_encode(s, cfg.block_size, pad_token, encode) for s in val_bad_sequences]
val_1_good_data = [pad_encode(s, cfg.block_size, pad_token, encode) for s in val_1_good_sequences]
val_1_bad_data = [pad_encode(s, cfg.block_size, pad_token, encode) for s in val_1_bad_sequences]

print(f"Project root: {project_root}")
print(f"Device: {device}")
print("Dataset sizes (good/bad):")
print(f"  train: {len(train_good_data)} / {len(train_bad_data)}")
print(f"  val:   {len(val_good_data)} / {len(val_bad_data)}")
print(f"  val_1: {len(val_1_good_data)} / {len(val_1_bad_data)}")

In [ ]:
# Inserisci qui tutti i path che vuoi confrontare.
history_paths = [

    ### CHANGE BETA ###
    # "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-100_beta_0.1/allpairs_nt_refbin_reciprocal/history.json",
    # "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-100/allpairs_bin_refbin_reciprocal/history.json",
    # "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-100_beta_0.5/allpairs_nt_refbin_reciprocal/history.json",
    # "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-100_beta_0.8/allpairs_nt_refbin_reciprocal/history.json",

    ### CHANGE FILTER ###
    # "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-100/allpairs_bin_refbin_reciprocal/history.json",
    # "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-90/allpairs_bin_refbin_reciprocal/history.json",
    # "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-80/allpairs_bin_refbin_reciprocal/history.json",
    # "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-70/allpairs_bin_refbin_reciprocal/history.json",
    # "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-60/allpairs_bin_refbin_reciprocal/history.json",

    ### Binary vs Nucleotides
    # "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-100/top100_bin/history.json",
    # "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-100/top100_nt/history.json",

    ### TOP K ###
    # "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-100/top5_bin/history.json",
    # "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-100/top20_bin/history.json",
    # "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-100/top100_bin/history.json",
    # "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-100/top500_bin/history.json",

    ### bin, refbin, reciprocal
    # "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-100/top100_bin/history.json",
    # "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-100/top100_bin_refbin/history.json",
    # "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-100/top100_bin_refbin_reciprocal/history.json",

    ### binwise vs full
    # "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-100/allpairs_bin_refbin_reciprocal/history.json",
    # "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-100/allpairs/history.json",


    ### compare final
    "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-100/allpairs_bin_refbin_reciprocal/history.json",
    "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-100/allpairs/history.json",
    "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-70/allpairs_bin_refbin_reciprocal/history.json",
    "../images_dpo/split_0.15_vae25-30-and-close-in-test_nll-filter-70/allpairs/history.json",
]


# Percorso per salvare i plot generati
output_path = Path("../output_plots/compare_final")  # Cambia nome se vuoi
output_path.mkdir(parents=True, exist_ok=True)
print(f"Output plots will be saved to: {output_path.resolve()}")


if len(history_paths) == 0:
    raise ValueError("Inserisci almeno un percorso in history_paths.")

In [ ]:
def _parse_constant(value: str):
    # I file possono contenere NaN/Infinity non strettamente JSON standard.
    mapping = {"NaN": float("nan"), "Infinity": float("inf"), "-Infinity": float("-inf")}
    if value in mapping:
        return mapping[value]
    raise ValueError(f"Costante non supportata nel JSON: {value}")


def load_history(path_str: str) -> pd.DataFrame:
    path = Path(path_str)
    if not path.exists():
        raise FileNotFoundError(f"File non trovato: {path}")

    with path.open("r", encoding="utf-8") as f:
        raw = json.load(f, parse_constant=_parse_constant)

    required = [
        "iteration",
        "train_dist_nll_corr",
        "val_dist_nll_corr",
        "val_1_dist_nll_corr",
        "val_sep_corr",
        "val_1_sep_corr",
    ]
    missing = [k for k in required if k not in raw]
    if missing:
        raise KeyError(f"Nel file {path} mancano le chiavi: {missing}")

    min_len = min(len(raw[k]) for k in required)
    if min_len == 0:
        raise ValueError(f"Nessun dato utile in {path}")

    return pd.DataFrame({
        "iteration": raw["iteration"][:min_len],
        "train": raw["train_dist_nll_corr"][:min_len],
        "validation": raw["val_dist_nll_corr"][:min_len],
        "validation 1": raw["val_1_dist_nll_corr"][:min_len],
        "val sep": raw["val_sep_corr"][:min_len],
        "val 1 sep": raw["val_1_sep_corr"][:min_len],
    })


dataframes = [load_history(p) for p in history_paths]
[df.head(3) for df in dataframes]

In [ ]:
# Carica le matrici NLL per split/label dai file history
seq_nll_keys = [
    "train_good_seq_nll",
    "train_bad_seq_nll",
    "val_good_seq_nll",
    "val_bad_seq_nll",
    "val_1_good_seq_nll",
    "val_1_bad_seq_nll",
]

seq_nll_data: dict[str, dict[str, np.ndarray]] = {k: {} for k in seq_nll_keys}


def history_run_label_from_abs(history_abs: Path) -> str:
    try:
        rel = history_abs.relative_to(project_root)
    except ValueError:
        rel = history_abs

    if len(rel.parts) >= 3 and rel.parts[0] == "images_dpo":
        split_name = rel.parts[1]
        pairing = rel.parts[-2]

        nll_match = re.search(r"nll-filter-(\d+)", split_name)
        nll_filter = nll_match.group(1) if nll_match else "100"
        nll_str = f"NLL:{nll_filter}" if nll_filter != "100" else ""

        beta_match = re.search(r"beta_([0-9.]+)", split_name)
        beta = beta_match.group(1) if beta_match else "0.3"
        beta_str = f"β:{beta}"

        params = [p for p in [nll_str, beta_str] if p]
        return f"{pairing} ({', '.join(params)})" if params else pairing

    return str(rel.parent if history_abs.name == "history.json" else rel)


for history_path_str in history_paths:
    history_abs = Path(history_path_str)
    if not history_abs.is_absolute():
        history_abs = (project_root / "notebooks" / history_abs).resolve()

    if not history_abs.exists():
        print(f"[WARN] History file non trovato: {history_abs}")
        continue

    with history_abs.open("r", encoding="utf-8") as f:
        raw = json.load(f, parse_constant=_parse_constant)

    run_label = history_run_label_from_abs(history_abs)
    print(f"Run: {run_label}")

    for key in seq_nll_keys:
        if key not in raw:
            print(f"  [WARN] chiave mancante: {key}")
            continue

        arr = np.array(raw[key], dtype=float)
        seq_nll_data[key][run_label] = arr
        print(f"  {key}: shape={arr.shape}, dtype={arr.dtype}")

    print()

# Alias utili per accesso rapido
train_good_seq_nll_data = seq_nll_data["train_good_seq_nll"]
train_bad_seq_nll_data = seq_nll_data["train_bad_seq_nll"]
val_bad_seq_nll_data = seq_nll_data["val_bad_seq_nll"]
val_1_good_seq_nll_data = seq_nll_data["val_1_good_seq_nll"]
val_1_bad_seq_nll_data = seq_nll_data["val_1_bad_seq_nll"]

print("Riepilogo run caricati per chiave:")
for key in seq_nll_keys:
    print(f"  {key}: {len(seq_nll_data[key])} run")

In [ ]:
metric_map = [
    ("train", "train_dist_nll_corr"),
    ("validation", "val_dist_nll_corr"),
    ("validation 1", "val_1_dist_nll_corr"),
    ("val sep", "val_sep_corr"),
    ("val 1 sep", "val_1_sep_corr"),
]

def history_run_label(path_str: str) -> str:
    history_path = Path(path_str)
    if not history_path.is_absolute():
        history_path = (project_root / "notebooks" / history_path).resolve()
    try:
        rel = history_path.relative_to(project_root)
    except ValueError:
        rel = history_path

    if len(rel.parts) >= 3 and rel.parts[0] == "images_dpo":
        split_name = rel.parts[1]
        pairing = rel.parts[-2]
        
        # Estrai NLL filter: il numero dopo 'nll-filter-'
        nll_match = re.search(r"nll-filter-(\d+)", split_name)
        nll_filter = nll_match.group(1) if nll_match else "100"
        nll_str = f"NLL:{nll_filter}" if nll_filter != "100" else ""
        
        # Estrai beta: se presente, altrimenti default 0.3
        beta_match = re.search(r"beta_([0-9.]+)", split_name)
        beta = beta_match.group(1) if beta_match else "0.3"
        beta_str = f"β:{beta}"
        
        # Costruisci label: pairing (parametri)
        params = [p for p in [nll_str, beta_str] if p]
        if params:
            return f"{pairing} ({', '.join(params)})"
        else:
            return pairing

    return str(rel.parent if history_path.name == "history.json" else rel)


run_labels = [history_run_label(p) for p in history_paths]
# Mappa colori condivisa tra tutti i grafici del notebook.
base_cmap = plt.get_cmap("tab20", max(len(run_labels), 1))
run_color_map = {run_label: base_cmap(i) for i, run_label in enumerate(run_labels)}

# Mappatura dai dataset ai titoli:
dataset_title_map = {
    "train": "Training Set",
    "validation": "Validation Set",
    "validation 1": "VAE 25-30",
}

# Figura 1: primi 3 grafici (prima riga)
fig1, axes1 = plt.subplots(1, 3, figsize=(24, 6), sharex=True, sharey=False)

for ax, (col, metric_name) in zip(axes1, metric_map[:3]):
    for path_str, df in zip(history_paths, dataframes):
        run_label = history_run_label(path_str)
        ax.plot(
            df["iteration"],
            df[col],
            label=run_label,
            linewidth=2.5,
            color=run_color_map[run_label],
        )

    ax.set_title(dataset_title_map[col], fontsize=18, fontweight='bold')
    ax.set_xlabel("iteration", fontsize=16)
    ax.set_ylabel("Pearson correlation", fontsize=16)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=14)

# Sincronizza y-scale per i primi 2 subplot (training e validation)
y_min = min(axes1[0].get_ylim()[0], axes1[1].get_ylim()[0])
y_max = max(axes1[0].get_ylim()[1], axes1[1].get_ylim()[1])
axes1[0].set_ylim(y_min, y_max)
axes1[1].set_ylim(y_min, y_max)

fig1.suptitle("Pearson NLL vs distance from Azoarcus", fontsize=20, fontweight='bold')
plt.tight_layout()
plt.savefig(output_path / "pearson_dist_nll.png", dpi=150, bbox_inches='tight')
print(f"Figura 1 salvata: {output_path / 'pearson_dist_nll.png'}")
plt.show()

# Figura 2: ultimi 2 grafici (seconda riga)
fig2, axes2 = plt.subplots(1, 2, figsize=(16, 6), sharex=False, sharey=False)

dataset_title_map_fig2 = {
    "val sep": "Validation Set",
    "val 1 sep": "VAE 25-30",
}

for ax, (col, metric_name) in zip(axes2, metric_map[3:]):
    for path_str, df in zip(history_paths, dataframes):
        run_label = history_run_label(path_str)
        ax.plot(
            df["iteration"],
            df[col],
            label=run_label,
            linewidth=2.5,
            color=run_color_map[run_label],
        )

    ax.set_title(dataset_title_map_fig2[col], fontsize=18, fontweight='bold')
    ax.set_xlabel("iteration", fontsize=16)
    ax.set_ylabel("Pearson correlation", fontsize=16)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=14)

# Sincronizza y-scale per entrambi i subplot di figura 2
y_min_fig2 = min(axes2[0].get_ylim()[0], axes2[1].get_ylim()[0])
y_max_fig2 = max(axes2[0].get_ylim()[1], axes2[1].get_ylim()[1])
axes2[0].set_ylim(y_min_fig2, y_max_fig2)
axes2[1].set_ylim(y_min_fig2, y_max_fig2)

fig2.suptitle("Pearson NLL vs Label", fontsize=20, fontweight='bold')
plt.tight_layout()
plt.savefig(output_path / "pearson_label_nll.png", dpi=150, bbox_inches='tight')
print(f"Figura 2 salvata: {output_path / 'pearson_label_nll.png'}")
plt.show()

In [ ]:
def build_model() -> GPTTransformer:
    model = GPTTransformer(
        vocab_size=vocab_size,
        embed_dim=cfg.n_embd,
        num_layers=cfg.n_layer,
        num_heads=cfg.n_head,
        mlp_ratio=4,
        dropout_p=0.1,
        pad_id=pad_token,
    )
    return model.to(device)


def resolve_history_path(path_str: str) -> Path:
    p = Path(path_str)
    if p.is_absolute():
        return p
    # I path nel notebook sono relativi alla cartella notebooks.
    return (project_root / "notebooks" / p).resolve()


def history_run_label(history_path: Path) -> str:
    history_path = history_path.resolve()
    try:
        rel = history_path.relative_to(project_root)
    except ValueError:
        rel = history_path

    if len(rel.parts) >= 3 and rel.parts[0] == "images_dpo":
        split_name = rel.parts[1]
        run_name = rel.parts[-2]
        return f"{split_name} | {run_name}"

    return str(rel.parent if history_path.name == "history.json" else rel)


def checkpoint_dir_from_history_path(history_path: Path) -> Path:
    rel = history_path.resolve().relative_to(project_root)
    if len(rel.parts) < 3 or rel.parts[0] != "images_dpo":
        raise ValueError(f"Path history non supportato per mapping checkpoint: {history_path}")
    return project_root / "checkpoints" / "checkpoints_dpo" / Path(*rel.parts[1:-1])


def list_checkpoint_iterations(ckpt_dir: Path) -> dict[int, Path]:
    pattern = re.compile(r"dpo_model_iter(\d+)\.pt$")
    found: dict[int, Path] = {}
    if not ckpt_dir.exists():
        return found
    for ckpt in ckpt_dir.glob("dpo_model_iter*.pt"):
        m = pattern.search(ckpt.name)
        if m is None:
            continue
        found[int(m.group(1))] = ckpt
    return dict(sorted(found.items(), key=lambda kv: kv[0]))


def _normalize_state_dict_keys(state_dict: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    # Alcuni checkpoint salvati con torch.compile hanno il prefisso '_orig_mod.'.
    if all(k.startswith("_orig_mod.") for k in state_dict.keys()):
        return {k.removeprefix("_orig_mod."): v for k, v in state_dict.items()}
    return state_dict


def compute_auroc_auprc_from_nll(good_nll: list[float], bad_nll: list[float]) -> tuple[float, float]:
    labels, scores = _build_binary_labels_and_scores(good_nll, bad_nll)
    _, _, auroc = _compute_roc_curve(labels, scores)
    _, _, auprc = _compute_pr_curve(labels, scores)
    return float(auroc), float(auprc)


def evaluate_checkpoint_for_splits(model_state_path: Path) -> dict[str, dict[str, float]]:
    model = build_model()
    state_dict = torch.load(model_state_path, map_location="cpu")
    state_dict = _normalize_state_dict_keys(state_dict)
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()

    with torch.inference_mode():
        train_good_nll = compute_sequence_nll(model, train_good_data, pad_token, device, batch_size=cfg.full_eval_batch_size_cap).tolist()
        train_bad_nll = compute_sequence_nll(model, train_bad_data, pad_token, device, batch_size=cfg.full_eval_batch_size_cap).tolist()
        val_good_nll = compute_sequence_nll(model, val_good_data, pad_token, device, batch_size=cfg.full_eval_batch_size_cap).tolist()
        val_bad_nll = compute_sequence_nll(model, val_bad_data, pad_token, device, batch_size=cfg.full_eval_batch_size_cap).tolist()
        val_1_good_nll = compute_sequence_nll(model, val_1_good_data, pad_token, device, batch_size=cfg.full_eval_batch_size_cap).tolist()
        val_1_bad_nll = compute_sequence_nll(model, val_1_bad_data, pad_token, device, batch_size=cfg.full_eval_batch_size_cap).tolist()

    return {
        "train": dict(zip(["auroc", "auprc"], compute_auroc_auprc_from_nll(train_good_nll, train_bad_nll))),
        "val": dict(zip(["auroc", "auprc"], compute_auroc_auprc_from_nll(val_good_nll, val_bad_nll))),
        "val_1": dict(zip(["auroc", "auprc"], compute_auroc_auprc_from_nll(val_1_good_nll, val_1_bad_nll))),
    }


# Baseline a iterazione 0: modello pretrained iniziale.
baseline_metrics = evaluate_checkpoint_for_splits(pretrained_ckpt_path)
baseline_metrics

In [ ]:
all_metric_rows: list[dict[str, object]] = []
missing_checkpoint_summary: dict[str, list[int]] = {}


def select_target_iterations(history_iterations: list[int], step: int = 5000) -> list[int]:
    if len(history_iterations) == 0:
        return []

    max_iter = max(history_iterations)
    targets = {
        it for it in history_iterations
        if it == 0 or it == max_iter or (it > 0 and it % step == 0)
    }
    return sorted(targets)


# 1) Baseline iterazione 0 calcolata una sola volta (gia' in baseline_metrics)
#    e replicata su tutti i run che hanno iteration=0 tra le iterazioni target.

for history_path_str in history_paths:
    history_abs = resolve_history_path(history_path_str)
    if not history_abs.exists():
        raise FileNotFoundError(f"History file non trovato: {history_abs}")

    with history_abs.open("r", encoding="utf-8") as f:
        history_obj = json.load(f)

    history_iterations = sorted({int(it) for it in history_obj.get("iteration", [])})
    target_iterations = select_target_iterations(history_iterations)
    run_name = history_run_label(history_abs)

    if 0 in target_iterations:
        for split_name in ["train", "val", "val_1"]:
            all_metric_rows.append(
                {
                    "run": run_name,
                    "history_path": str(history_abs),
                    "checkpoint_dir": str(checkpoint_dir_from_history_path(history_abs)),
                    "iteration": 0,
                    "split": split_name,
                    "auroc": baseline_metrics[split_name]["auroc"],
                    "auprc": baseline_metrics[split_name]["auprc"],
                }
            )

# 2) Per ogni run valuta tutti i checkpoint target: multipli di 5000 e ultima iterazione.
for history_path_str in history_paths:
    history_abs = resolve_history_path(history_path_str)
    if not history_abs.exists():
        raise FileNotFoundError(f"History file non trovato: {history_abs}")

    with history_abs.open("r", encoding="utf-8") as f:
        history_obj = json.load(f)

    history_iterations = sorted({int(it) for it in history_obj.get("iteration", [])})
    target_iterations = [it for it in select_target_iterations(history_iterations) if it != 0]
    if len(target_iterations) == 0:
        continue

    ckpt_dir = checkpoint_dir_from_history_path(history_abs)
    ckpt_by_iter = list_checkpoint_iterations(ckpt_dir)
    run_name = history_run_label(history_abs)
    missing_iters: list[int] = []

    for target_iter in target_iterations:
        ckpt_path = ckpt_by_iter.get(target_iter)
        if ckpt_path is None:
            missing_iters.append(target_iter)
            continue

        split_metrics = evaluate_checkpoint_for_splits(ckpt_path)
        for split_name in ["train", "val", "val_1"]:
            all_metric_rows.append(
                {
                    "run": run_name,
                    "history_path": str(history_abs),
                    "checkpoint_dir": str(ckpt_dir),
                    "iteration": target_iter,
                    "split": split_name,
                    "auroc": split_metrics[split_name]["auroc"],
                    "auprc": split_metrics[split_name]["auprc"],
                }
            )

    if len(missing_iters) > 0:
        missing_checkpoint_summary[run_name] = missing_iters

auroc_auprc_df = pd.DataFrame(all_metric_rows).sort_values(["run", "iteration", "split"]).reset_index(drop=True)

print(f"Totale righe metriche calcolate: {len(auroc_auprc_df)}")
if len(missing_checkpoint_summary) > 0:
    print("Iterazioni target senza checkpoint corrispondente:")
    for run_name, miss in missing_checkpoint_summary.items():
        print(f"  {run_name}: {miss}")

auroc_auprc_df.head(12)

In [ ]:
if "auroc_auprc_df" not in globals() or auroc_auprc_df.empty:
    raise RuntimeError("Nessuna metrica disponibile. Esegui prima la cella precedente.")

# Ricrea history_run_label con la stessa logica della cella 6
def history_run_label_corrected(path_str: str) -> str:
    history_path = Path(path_str)
    if not history_path.is_absolute():
        history_path = (project_root / "notebooks" / history_path).resolve()
    try:
        rel = history_path.relative_to(project_root)
    except ValueError:
        rel = history_path

    if len(rel.parts) >= 3 and rel.parts[0] == "images_dpo":
        split_name = rel.parts[1]
        pairing = rel.parts[-2]

        nll_match = re.search(r"nll-filter-(\d+)", split_name)
        nll_filter = nll_match.group(1) if nll_match else "100"
        nll_str = f"NLL:{nll_filter}" if nll_filter != "100" else ""

        beta_match = re.search(r"beta_([0-9.]+)", split_name)
        beta = beta_match.group(1) if beta_match else "0.3"
        beta_str = f"β:{beta}"

        params = [p for p in [nll_str, beta_str] if p]
        if params:
            return f"{pairing} ({', '.join(params)})"
        else:
            return pairing

    return str(rel.parent if history_path.name == "history.json" else rel)

# Crea mapping da run_labels vecchi a nuovi, basato sulla history_path
old_to_new_run = {}
for history_path_str in history_paths:
    history_abs = Path(history_path_str)
    if not history_abs.is_absolute():
        history_abs = (project_root / "notebooks" / history_abs).resolve()
    new_label = history_run_label_corrected(history_path_str)

    # Trova il run_label vecchio dal dataframe
    rows_with_path = auroc_auprc_df[auroc_auprc_df["history_path"] == str(history_abs)]
    if not rows_with_path.empty:
        old_label = rows_with_path["run"].iloc[0]
        old_to_new_run[old_label] = new_label

# Aggiorna il dataframe con i run_labels corretti
auroc_auprc_df["run"] = auroc_auprc_df["run"].map(old_to_new_run).fillna(auroc_auprc_df["run"])

# Mappa colori coerente con le figure 1-2: stesso ordine di history_paths
run_labels_consistent = [history_run_label_corrected(p) for p in history_paths]
base_cmap_consistent = plt.get_cmap("tab20", max(len(run_labels_consistent), 1))
run_color_map = {run_label: base_cmap_consistent(i) for i, run_label in enumerate(run_labels_consistent)}

splits = ["train", "val", "val_1"]
fig, axes = plt.subplots(2, 3, figsize=(28, 12), sharex=True, sharey=False)

# Mappatura dai dataset ai titoli
dataset_title_map = {
    "train": "Training Set",
    "val": "Validation Set",
    "val_1": "VAE 25-30",
}

# Riga AUROC
for col_idx, split_name in enumerate(splits):
    ax = axes[0, col_idx]
    sub = auroc_auprc_df[auroc_auprc_df["split"] == split_name]
    for run_name, run_df in sub.groupby("run", sort=False):
        ax.plot(
            run_df["iteration"],
            run_df["auroc"],
            linewidth=2.5,
            marker="o",
            markersize=4,
            label=run_name,
            color=run_color_map.get(run_name),
        )
    ax.set_title(dataset_title_map[split_name], fontsize=18, fontweight='bold')
    if col_idx == 0:
        ax.set_ylabel("AUROC", fontsize=16, fontweight='bold')
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=14)

# Riga AUPRC
for col_idx, split_name in enumerate(splits):
    ax = axes[1, col_idx]
    sub = auroc_auprc_df[auroc_auprc_df["split"] == split_name]
    for run_name, run_df in sub.groupby("run", sort=False):
        ax.plot(
            run_df["iteration"],
            run_df["auprc"],
            linewidth=2.5,
            marker="o",
            markersize=4,
            label=run_name,
            color=run_color_map.get(run_name),
        )
    ax.set_title(dataset_title_map[split_name], fontsize=18, fontweight='bold')
    ax.set_xlabel("iteration", fontsize=16)
    if col_idx == 0:
        ax.set_ylabel("AUPRC", fontsize=16, fontweight='bold')
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=14)

fig.suptitle("Confronto checkpoint per iterazione: AUROC/AUPRC su train, val, val_1", fontsize=20, fontweight='bold')
plt.tight_layout()
plt.savefig(output_path / "auroc_auprc.png", dpi=150, bbox_inches='tight')
print(f"Figura 3 salvata: {output_path / 'auroc_auprc.png'}")
plt.show()

In [ ]:
# Violin plot multi-subplot: Training / Validation / VAE 25-30
if "seq_nll_data" not in globals():
    raise RuntimeError("seq_nll_data non disponibile. Esegui prima la cella di caricamento NLL.")

if "history_paths" not in globals() or len(history_paths) == 0:
    raise RuntimeError("history_paths non definito o vuoto.")


def _run_label_from_abs_path(history_abs: Path) -> str:
    try:
        rel = history_abs.relative_to(project_root)
    except ValueError:
        rel = history_abs

    if len(rel.parts) >= 3 and rel.parts[0] == "images_dpo":
        split_name = rel.parts[1]
        pairing = rel.parts[-2]

        nll_match = re.search(r"nll-filter-(\d+)", split_name)
        nll_filter = nll_match.group(1) if nll_match else "100"
        nll_str = f"NLL:{nll_filter}" if nll_filter != "100" else ""

        beta_match = re.search(r"beta_([0-9.]+)", split_name)
        beta = beta_match.group(1) if beta_match else "0.3"
        beta_str = f"β:{beta}"

        params = [p for p in [nll_str, beta_str] if p]
        return f"{pairing} ({', '.join(params)})" if params else pairing

    return str(rel.parent if history_abs.name == "history.json" else rel)


def _resolve_history(path_str: str) -> Path:
    p = Path(path_str)
    if p.is_absolute():
        return p
    return (project_root / "notebooks" / p).resolve()


def _build_plot_records(good_key: str, bad_key: str) -> list[dict[str, object]]:
    records: list[dict[str, object]] = []

    for hp in history_paths:
        history_abs = _resolve_history(hp)
        if not history_abs.exists():
            print(f"[WARN] History file non trovato: {history_abs}")
            continue

        run_label = _run_label_from_abs_path(history_abs)

        good_arr = seq_nll_data.get(good_key, {}).get(run_label)
        bad_arr = seq_nll_data.get(bad_key, {}).get(run_label)

        if good_arr is None or bad_arr is None:
            print(f"[WARN] {good_key}/{bad_key} mancanti per run: {run_label}")
            continue

        with history_abs.open("r", encoding="utf-8") as f:
            raw = json.load(f, parse_constant=_parse_constant)

        iterations = [int(x) for x in raw.get("iteration", [])]
        if len(iterations) == 0:
            print(f"[WARN] iteration vuoto in: {history_abs}")
            continue

        usable_len = min(len(iterations), good_arr.shape[0], bad_arr.shape[0])
        iterations = iterations[:usable_len]
        good_arr = good_arr[:usable_len]
        bad_arr = bad_arr[:usable_len]

        selected_idx = [i for i, it in enumerate(iterations) if (it == 0 or it % 5000 == 0)]
        if len(selected_idx) == 0:
            print(f"[WARN] Nessuna iterazione selezionata per run: {run_label}")
            continue

        records.append(
            {
                "run_label": run_label,
                "iterations": [iterations[i] for i in selected_idx],
                "good_vectors": [good_arr[i] for i in selected_idx],
                "bad_vectors": [bad_arr[i] for i in selected_idx],
            }
        )

    return records


plot_specs = [
    ("train_good_seq_nll", "train_bad_seq_nll", "Training Set"),
    ("val_good_seq_nll", "val_bad_seq_nll", "Validation Set"),
    ("val_1_good_seq_nll", "val_1_bad_seq_nll", "VAE 25-30"),
]

n_runs = len(history_paths)
greens = plt.get_cmap("Greens")
reds = plt.get_cmap("Reds")

# Evita tonalita troppo chiare
good_colors = [greens(0.45 + 0.45 * (i / max(n_runs - 1, 1))) for i in range(n_runs)]
bad_colors = [reds(0.45 + 0.45 * (i / max(n_runs - 1, 1))) for i in range(n_runs)]

fig, axes = plt.subplots(3, 1, figsize=(26, 22), sharex=False)

run_offsets = np.linspace(-560, 560, max(n_runs, 1))
group_shift = 1000
violin_width = 950

legend_handles = []
legend_labels = []

for ax_idx, (good_key, bad_key, subplot_title) in enumerate(plot_specs):
    ax = axes[ax_idx]
    plot_records = _build_plot_records(good_key, bad_key)

    if len(plot_records) == 0:
        ax.set_title(f"{subplot_title} (nessun dato)", fontsize=17, fontweight="bold")
        ax.grid(True, alpha=0.25)
        continue

    # Mappa run -> indice colore/offset usando l'ordine di history_paths
    run_to_idx: dict[str, int] = {}
    for idx, hp in enumerate(history_paths):
        history_abs = _resolve_history(hp)
        if history_abs.exists():
            run_to_idx[_run_label_from_abs_path(history_abs)] = idx

    for rec in plot_records:
        run_label = rec["run_label"]
        run_idx = run_to_idx.get(run_label, 0)

        iters = rec["iterations"]
        good_vectors = rec["good_vectors"]
        bad_vectors = rec["bad_vectors"]

        good_pos = np.array(iters, dtype=float) - group_shift + run_offsets[run_idx]
        bad_pos = np.array(iters, dtype=float) + group_shift + run_offsets[run_idx]

        vp_good = ax.violinplot(good_vectors, positions=good_pos, widths=violin_width, showmeans=False, showextrema=False)
        vp_bad = ax.violinplot(bad_vectors, positions=bad_pos, widths=violin_width, showmeans=False, showextrema=False)

        for body in vp_good["bodies"]:
            body.set_facecolor(good_colors[run_idx])
            body.set_edgecolor(good_colors[run_idx])
            body.set_alpha(0.55)

        for body in vp_bad["bodies"]:
            body.set_facecolor(bad_colors[run_idx])
            body.set_edgecolor(bad_colors[run_idx])
            body.set_alpha(0.55)

    all_iters = sorted({it for rec in plot_records for it in rec["iterations"]})
    ax.set_xticks(all_iters)
    ax.set_xticklabels([str(it) for it in all_iters], rotation=45, ha="right", fontsize=11)
    ax.set_ylabel("Sequence NLL", fontsize=15)
    ax.set_title(subplot_title, fontsize=17, fontweight="bold")
    ax.grid(True, alpha=0.25)

    # Costruisci legenda una sola volta (dal primo subplot)
    if ax_idx == 0:
        for run_label, run_idx in sorted(run_to_idx.items(), key=lambda x: x[1]):
            h_good = plt.Line2D([0], [0], color=good_colors[run_idx], lw=8)
            h_bad = plt.Line2D([0], [0], color=bad_colors[run_idx], lw=8)
            legend_handles.extend([h_good, h_bad])
            legend_labels.extend([f"GOOD - {run_label}", f"BAD - {run_label}"])

axes[-1].set_xlabel("Iteration", fontsize=16)
fig.suptitle("Violin NLL per split: good (verde) vs bad (rosso)", fontsize=22, fontweight="bold")

if len(legend_handles) > 0:
    fig.legend(legend_handles, legend_labels, loc="upper right", fontsize=9, ncol=2)

plt.tight_layout(rect=[0, 0, 1, 0.97])
save_path = output_path / "train_val_vae_violin_green_red_scales.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"Figura salvata: {save_path}")
plt.show()

In [ ]:
# Correlazione NLL vs distanza da DN[0] su DT+_197 e DT-_197
# (da cella 14 in poi)
from math import isnan

# 1) Carica dataset richiesti
plus_path_197 = project_root / "data" / "DT+_197.fasta"
minus_path_197 = project_root / "data" / "DT-_197.fasta"
dn_path_197 = project_root / "data" / "DN_197.fasta"

for p in [plus_path_197, minus_path_197, dn_path_197]:
    if not p.exists():
        raise FileNotFoundError(f"File non trovato: {p}")

plus_sequences_197 = read_fasta(plus_path_197)
minus_sequences_197 = read_fasta(minus_path_197)
dn_sequences_197 = read_fasta(dn_path_197)

if len(dn_sequences_197) == 0:
    raise RuntimeError("DN_197.fasta e' vuoto.")

ref_dn_seq = dn_sequences_197[0]

# 2) Distanza (Hamming estesa: mismatch + differenza di lunghezza)
def distance_to_ref(seq: str, ref: str) -> int:
    min_len = min(len(seq), len(ref))
    mismatches = sum(1 for a, b in zip(seq[:min_len], ref[:min_len]) if a != b)
    return mismatches + abs(len(seq) - len(ref))

plus_dist = np.array([distance_to_ref(s, ref_dn_seq) for s in plus_sequences_197], dtype=float)
minus_dist = np.array([distance_to_ref(s, ref_dn_seq) for s in minus_sequences_197], dtype=float)

# 3) Encoding per NLL con pipeline gia' usata nel notebook
plus_data_197 = [pad_encode(s, cfg.block_size, pad_token, encode) for s in plus_sequences_197]
minus_data_197 = [pad_encode(s, cfg.block_size, pad_token, encode) for s in minus_sequences_197]

if len(plus_data_197) != len(plus_dist) or len(minus_data_197) != len(minus_dist):
    raise RuntimeError("Lunghezze incoerenti tra sequenze e distanze.")

# 4) Per ogni history path e per ogni iterazione multipla di 5000 calcola correlazione NLL-distanza
rows_corr: list[dict[str, object]] = []

for hp in history_paths:
    history_abs = resolve_history_path(hp)
    if not history_abs.exists():
        print(f"[WARN] History non trovato: {history_abs}")
        continue

    # Label coerente con le prime figure
    run_label = history_run_label(history_abs) if "history_run_label" in globals() else str(history_abs)

    ckpt_dir = checkpoint_dir_from_history_path(history_abs)
    ckpt_by_iter = list_checkpoint_iterations(ckpt_dir)

    target_iters = sorted(it for it in ckpt_by_iter.keys() if it % 5000 == 0)
    if len(target_iters) == 0:
        print(f"[WARN] Nessun checkpoint multiplo di 5000 per run: {run_label}")
        continue

    print(f"Run: {run_label} | iterazioni target: {target_iters}")

    model = build_model()

    for it in target_iters:
        ckpt_path = ckpt_by_iter[it]

        state_dict = torch.load(ckpt_path, map_location="cpu")
        state_dict = _normalize_state_dict_keys(state_dict)
        model.load_state_dict(state_dict)
        model = model.to(device)
        model.eval()

        with torch.inference_mode():
            plus_nll = np.array(
                compute_sequence_nll(
                    model,
                    plus_data_197,
                    pad_token,
                    device,
                    batch_size=cfg.full_eval_batch_size_cap,
                ).tolist(),
                dtype=float,
            )
            minus_nll = np.array(
                compute_sequence_nll(
                    model,
                    minus_data_197,
                    pad_token,
                    device,
                    batch_size=cfg.full_eval_batch_size_cap,
                ).tolist(),
                dtype=float,
            )

        all_nll = np.concatenate([plus_nll, minus_nll])
        all_dist = np.concatenate([plus_dist, minus_dist])

        if np.std(all_nll) == 0 or np.std(all_dist) == 0:
            corr = float("nan")
        else:
            corr = float(np.corrcoef(all_nll, all_dist)[0, 1])

        rows_corr.append(
            {
                "run": run_label,
                "iteration": int(it),
                "corr_nll_distance": corr,
                "history_path": str(history_abs),
            }
        )

corr_df = pd.DataFrame(rows_corr).sort_values(["run", "iteration"]).reset_index(drop=True)

if corr_df.empty:
    raise RuntimeError("Nessuna correlazione calcolata: verifica history_paths/checkpoint.")

print("\nPrime righe correlazioni:")
print(corr_df.head(12))

# 5) Plot: iteration (x) vs correlazione (y), legenda = history paths
fig, ax = plt.subplots(figsize=(16, 7))

for run_name, sub in corr_df.groupby("run", sort=False):
    color = run_color_map.get(run_name) if "run_color_map" in globals() else None
    ax.plot(
        sub["iteration"],
        sub["corr_nll_distance"],
        marker="o",
        linewidth=2.5,
        markersize=5,
        label=run_name,
        color=color,
    )

ax.set_xlabel("iteration", fontsize=14)
ax.set_ylabel("Pearson corr(NLL, distance)", fontsize=14)
ax.set_title("Correlazione NLL-distanza da DN[0] su DT+_197 e DT-_197", fontsize=17, fontweight="bold")
ax.grid(True, alpha=0.25)
ax.legend(fontsize=10)

plt.tight_layout()
save_path = output_path / "corr_nll_distance_dtp_dtm_197.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"Figura salvata: {save_path}")
plt.show()

In [ ]:
# Pearson NLL vs label su checkpoint multipli di 5000
if "history_paths" not in globals() or len(history_paths) == 0:
    raise RuntimeError("history_paths non definito o vuoto.")

# 1) Carica dataset nll-filtered_100
dtp_100_val_path = project_root / "data/split_train_validation/split_fraction-0.15_add-vae-25-30-and-close-to-validation_nll-filtered_100/DTp_val_plus-vae25-30-and-close.fasta"
dtm_100_val_path = project_root / "data/split_train_validation/split_fraction-0.15_add-vae-25-30-and-close-to-validation_nll-filtered_100/DTm_val_plus-vae25-30-and-close.fasta"

for p in [dtp_100_val_path, dtm_100_val_path]:
    if not p.exists():
        raise FileNotFoundError(f"File non trovato: {p}")

DTp_100_val_sequences = read_fasta(dtp_100_val_path)
DTm_100_val_sequences = read_fasta(dtm_100_val_path)

if len(DTp_100_val_sequences) == 0 or len(DTm_100_val_sequences) == 0:
    raise RuntimeError("Uno tra DTp_100_val o DTm_100_val e' vuoto.")

DTp_100_val_data = [pad_encode(s, cfg.block_size, pad_token, encode) for s in DTp_100_val_sequences]
DTm_100_val_data = [pad_encode(s, cfg.block_size, pad_token, encode) for s in DTm_100_val_sequences]

print(f"DTp_100_val size: {len(DTp_100_val_data)}")
print(f"DTm_100_val size: {len(DTm_100_val_data)}")

# 2) Carica dataset nll-filtered_70
dtp_70_val_path = project_root / "data/split_train_validation/split_fraction-0.15_add-vae-25-30-and-close-to-validation_nll-filtered_70/DTp_val_plus-vae25-30-and-close.fasta"
dtm_70_val_path = project_root / "data/split_train_validation/split_fraction-0.15_add-vae-25-30-and-close-to-validation_nll-filtered_70/DTm_val_plus-vae25-30-and-close.fasta"

for p in [dtp_70_val_path, dtm_70_val_path]:
    if not p.exists():
        raise FileNotFoundError(f"File non trovato: {p}")

DTp_70_val_sequences = read_fasta(dtp_70_val_path)
DTm_70_val_sequences = read_fasta(dtm_70_val_path)

if len(DTp_70_val_sequences) == 0 or len(DTm_70_val_sequences) == 0:
    raise RuntimeError("Uno tra DTp_70_val o DTm_70_val e' vuoto.")

DTp_70_val_data = [pad_encode(s, cfg.block_size, pad_token, encode) for s in DTp_70_val_sequences]
DTm_70_val_data = [pad_encode(s, cfg.block_size, pad_token, encode) for s in DTm_70_val_sequences]

print(f"DTp_70_val size: {len(DTp_70_val_data)}")
print(f"DTm_70_val size: {len(DTm_70_val_data)}")


def compute_binary_pearson_for_dataset(
    dataset_name: str,
    good_data: list[list[int]],
    bad_data: list[list[int]],
) -> pd.DataFrame:
    labels_binary = np.concatenate([
        np.zeros(len(good_data), dtype=float),
        np.ones(len(bad_data), dtype=float),
    ])

    if np.std(labels_binary) == 0:
        raise RuntimeError(f"Varianza etichette nulla per {dataset_name}: impossibile calcolare Pearson.")

    rows_binary_corr: list[dict[str, object]] = []

    for hp in history_paths:
        history_abs = resolve_history_path(hp)
        if not history_abs.exists():
            print(f"[WARN] History non trovato: {history_abs}")
            continue

        run_label = history_run_label(history_abs) if "history_run_label" in globals() else str(history_abs)
        ckpt_dir = checkpoint_dir_from_history_path(history_abs)
        ckpt_by_iter = list_checkpoint_iterations(ckpt_dir)

        # Multipli di 5000 presenti nei checkpoint del run
        target_iters = sorted(it for it in ckpt_by_iter.keys() if it % 5000 == 0)

        # Aggiungi baseline pretrained a iterazione 0
        iter_to_ckpt: dict[int, Path] = {0: pretrained_ckpt_path}
        iter_to_ckpt.update({it: ckpt_by_iter[it] for it in target_iters})

        if len(iter_to_ckpt) == 0:
            print(f"[WARN] Nessun checkpoint disponibile per run: {run_label}")
            continue

        print(f"[{dataset_name}] Run: {run_label} | iterazioni: {sorted(iter_to_ckpt.keys())}")
        model = build_model()

        for it in sorted(iter_to_ckpt.keys()):
            ckpt_path = iter_to_ckpt[it]
            state_dict = torch.load(ckpt_path, map_location="cpu")
            state_dict = _normalize_state_dict_keys(state_dict)
            model.load_state_dict(state_dict)
            model = model.to(device)
            model.eval()

            with torch.inference_mode():
                good_nll = np.array(
                    compute_sequence_nll(
                        model,
                        good_data,
                        pad_token,
                        device,
                        batch_size=cfg.full_eval_batch_size_cap,
                    ).tolist(),
                    dtype=float,
                )
                bad_nll = np.array(
                    compute_sequence_nll(
                        model,
                        bad_data,
                        pad_token,
                        device,
                        batch_size=cfg.full_eval_batch_size_cap,
                    ).tolist(),
                    dtype=float,
                )

            nll_all = np.concatenate([good_nll, bad_nll])
            if np.std(nll_all) == 0:
                pearson = float("nan")
            else:
                pearson = float(np.corrcoef(nll_all, labels_binary)[0, 1])

            rows_binary_corr.append(
                {
                    "dataset": dataset_name,
                    "run": run_label,
                    "iteration": int(it),
                    "pearson_nll_label": pearson,
                    "history_path": str(history_abs),
                }
            )

    df = pd.DataFrame(rows_binary_corr)
    if df.empty:
        raise RuntimeError(f"Nessuna correlazione calcolata per {dataset_name}: verifica history_paths/checkpoint.")
    return df.sort_values(["run", "iteration"]).reset_index(drop=True)


binary_corr_100_df = compute_binary_pearson_for_dataset(
    dataset_name="DTp_100_val (0) vs DTm_100_val (1)",
    good_data=DTp_100_val_data,
    bad_data=DTm_100_val_data,
)

binary_corr_70_df = compute_binary_pearson_for_dataset(
    dataset_name="DTp_70_val (0) vs DTm_70_val (1)",
    good_data=DTp_70_val_data,
    bad_data=DTm_70_val_data,
)

print("\nPrime righe correlazioni (100):")
print(binary_corr_100_df.head(8))
print("\nPrime righe correlazioni (70):")
print(binary_corr_70_df.head(8))

# 3) Figura unica con 2 subplot: sinistra=100, destra=70
fig, axes = plt.subplots(1, 2, figsize=(24, 7), sharex=False, sharey=True)

subplot_specs = [
    (axes[0], binary_corr_100_df, "DTp_100_val (0) vs DTm_100_val (1)"),
    (axes[1], binary_corr_70_df, "DTp_70_val (0) vs DTm_70_val (1)"),
]

for ax, df_plot, title in subplot_specs:
    for run_name, sub in df_plot.groupby("run", sort=False):
        color = run_color_map.get(run_name) if "run_color_map" in globals() else None
        ax.plot(
            sub["iteration"],
            sub["pearson_nll_label"],
            marker="o",
            linewidth=2.5,
            markersize=5,
            label=run_name,
            color=color,
        )

    ax.set_xlabel("iteration", fontsize=13)
    ax.set_title(title, fontsize=15, fontweight="bold")
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=9)

axes[0].set_ylabel("Pearson corr(NLL, label)", fontsize=13)
fig.suptitle("Pearson NLL-label per iterazione (checkpoint multipli di 5000)", fontsize=18, fontweight="bold")

plt.tight_layout(rect=[0, 0, 1, 0.96])
save_path = output_path / "corr_nll_label_dtp_dtm_val_100_70.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"Figura salvata: {save_path}")
plt.show()